In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
import pandas as pd

df1 = pd.read_csv('factory_min_real_final.csv')
df = df1.copy()

In [ ]:
df.info()

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt

df['localtime'] = pd.to_datetime(df['localtime'])

# 1. 클러스터 수동 할당
cluster_map = {
    '14(2호기)': 0, '18(우측분전반2)': 0,
    '123(3호기)': 1, '15(예비건조기)': 1, '17(6호기)': 1, '4(분쇄기(1))': 1, '5(좌측분전반)': 1,
    '1(PM-3)': 2, '4호기': 2, '16(호이스트)': 2, '3(분쇄기(2))': 2,
    '11(우측분전반1)': 3, '2(L-1전등)': 3
}
df['cluster'] = df['module(equipment)'].map(cluster_map)

# 2. 피처 정의 (모든 컬럼 사용, ID와 시간, 타겟 제외)
exclude_cols = ['module(equipment)', 'localtime', 'hourly_pow', 'cluster']
final_features = [col for col in df.columns if col not in exclude_cols]

print(f"학습에 사용될 피처 개수: {len(final_features)}개")

In [ ]:
def train_by_cluster(cluster_id, full_df):
    print(f"\n>>> Cluster {cluster_id} 학습 시작...")
    
    # 해당 클러스터 데이터만 추출
    c_df = full_df[full_df['cluster'] == cluster_id].copy()
    c_df = c_df.sort_values(['module(equipment)', 'localtime'])
    
    # 훈련/검증 분할 (마지막 7일 기준)
    split_date = c_df['localtime'].max() - pd.Timedelta(days=7)
    train = c_df[c_df['localtime'] < split_date]
    test = c_df[c_df['localtime'] >= split_date]
    
    X_train, y_train = train[final_features], train['hourly_pow']
    X_test, y_test = test[final_features], test['hourly_pow']
    
    # 클러스터별 맞춤 전략
    # 1, 2번(변동/피크)은 더 깊게(depth 10), 0, 3번은 기본(depth 6~8)
    if cluster_id in [1, 2]:
        curr_depth = 10
        curr_loss = 'MAE' # 변동성을 잡기 위해 중앙값 추적
    else:
        curr_depth = 8
        curr_loss = 'RMSE'
        
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=curr_depth,
        loss_function=curr_loss,
        eval_metric='SMAPE',
        random_seed=42,
        early_stopping_rounds=100,
        verbose=200,
        task_type="CPU" # GPU 가능하면 "GPU"로 변경
    )
    
    model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
    
    # 예측값 생성
    preds = model.predict(X_test)
    
    return y_test.values, preds, model

# 모든 클러스터 루프 실행
results = {}
for i in range(4):
    results[i] = train_by_cluster(i, df)